# 017 Matched PCA Baseline for MS-AA

This notebook provides a **targeted matched PCA comparison** for the MS-AA manuscript revision.

The goal is not to reproduce the full PCA/compressibility analysis from prior work. Instead, this notebook asks the reviewer-relevant question:

> How much of the current MS-AA result reflects dimensionality reduction generally, and what is specifically gained by the archetypal representation?

The PCA baseline deliberately mirrors the current MS-AA pipeline as closely as possible:

- same intact / word-scrambled / rest data;
- same HTFA representation;
- same trimming and normalization;
- same across-condition shared-basis logic;
- same spatial and temporal orientations;
- same cross-subject timepoint-matching metric;
- same model-order \(K\) framework;
- same sparse reconstruction idea.

Two PCA sparse baselines are included:

1. **variance-ranked PCA**, using the first \(m\) PCs, which is the natural PCA ordering and avoids decoding-based selection;
2. **optional decoding-ranked PCA**, provided only as a diagnostic comparison to the current AA top-\(m\) analysis.

For the paper, the variance-ranked PCA analysis is the cleaner primary comparison. If the AA top-\(m\) analysis is revised to use held-out ranking, PCA should also be evaluated under the same held-out selection framework.


In [ ]:
# ============================================================
# SETTINGS
# ============================================================

from pathlib import Path

# Analysis
FIT_SCOPE = "across"  # this benchmark is designed primarily for the across-condition paper analysis
ANALYSIS_TYPES = ["spatial", "temporal"]
CONDITIONS = ["intact", "word", "rest"]

# Use K values already present in the AA analysis.
# Edit these to exactly match the final paper.
K_VALUES_BY_ANALYSIS = {
    "spatial": [
        5, 7, 10, 14, 21, 25, 35, 40, 42, 44, 45, 46, 48, 50,
        52, 53, 54, 55, 56, 58, 60, 63, 65, 70, 75, 88, 100,
        105, 126, 140, 175, 189, 200, 210, 252, 300, 400, 500, 600, 700
    ],
    "temporal": [
        2, 3, 5, 6, 8, 9, 10, 11, 14, 15, 17, 18, 20, 23, 25,
        27, 30, 35, 38, 40, 45, 50, 54, 55, 60, 65, 75, 81, 90,
        100, 108, 200, 300
    ],
}

TOP_M_VALUES = [1, 2, 3, 5, 7, 10, 15, 20, 25, 30, 40, 50, 75, 100]

# Data settings copied from notebook 001
PIEMAN_MAT_PATH = "data/pieman/raw/pieman_data.mat"
TRIM_REST_BY = 100
HYPERTOOLS_NORMALIZE_PATH = "hypertools/normalize.py"

# Decoder settings copied from notebook 002
NFOLDS_DECODE = 2
NREPS_DECODE = 20
RNG_SEED = 42

# PCA
PCA_SOLVER = "full"
WHITEN = False

# Outputs
OUTPUT_DIR = Path("pca_matched_baseline_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FIG_ROOT = Path("/Users/lowen/Desktop/papers/archetypes/figures")
FIG_DIR = FIG_ROOT / "017_matched_pca_baseline"
FIG_DIR.mkdir(parents=True, exist_ok=True)

SAVE_FIGS = True
FIG_FORMAT = "pdf"
DPI = 300

# Optional comparison with existing AA outputs
AA_OUTPUT_TEMPLATE = "msaa_condrank_decoding_outputs_{analysis_type}_across"

# Natural PCA sparse ordering
RUN_VARIANCE_RANKED_TOPM = True

# Diagnostic only. This reproduces the logic of the current AA top-m analysis,
# but has the same selection/evaluation dependence that reviewers flagged.
RUN_DECODING_RANKED_PCA = False

# Raw HTFA baseline
RUN_RAW_HTFA_BASELINE = True

print("Output directory:", OUTPUT_DIR.resolve())
print("Figure directory:", FIG_DIR)


In [ ]:
# ============================================================
# IMPORTS
# ============================================================

%matplotlib inline

import os
import importlib.util
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.io import loadmat
from scipy.spatial.distance import cdist
from sklearn.decomposition import PCA

plt.rcParams.update({
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

_fig_counter = 0

def save_current_fig(name):
    global _fig_counter
    if not SAVE_FIGS:
        return
    _fig_counter += 1
    safe = "".join(c if c.isalnum() or c in "_-." else "_" for c in str(name))
    path = FIG_DIR / f"{_fig_counter:03d}_{safe}.{FIG_FORMAT}"
    plt.savefig(path, dpi=DPI, bbox_inches="tight")
    print("Saved:", path)


## Load and normalize the same HTFA data used by MS-AA

In [ ]:
# ============================================================
# DATA LOADING / NORMALIZATION
# copied closely from notebook 001
# ============================================================

HT_NORMALIZE_AVAILABLE = False
try:
    spec = importlib.util.spec_from_file_location("normalize", HYPERTOOLS_NORMALIZE_PATH)
    ht = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(ht)
    HT_NORMALIZE_AVAILABLE = True
except Exception:
    ht = None

def to_float_array(x):
    return np.asarray(x, dtype=float)

def array_cutter(data_list, cut_length):
    out = [to_float_array(x).copy() for x in data_list]
    return out if cut_length <= 0 else [x[:-cut_length] for x in out]

def coerce_TxV(x):
    x = to_float_array(x)
    if x.ndim != 2:
        raise ValueError(f"Expected 2D array, got {x.shape}")
    T, V = x.shape
    if V < T:
        x = x.T
    x = x - x.mean(axis=0, keepdims=True)
    return x.astype(np.float32, copy=False)

def normalize_across_fallback(subject_list, eps=1e-8):
    subj = [coerce_TxV(x) for x in subject_list]
    stacked = np.vstack(subj)
    mu = stacked.mean(axis=0, keepdims=True)
    sd = stacked.std(axis=0, keepdims=True) + eps
    return [((x - mu) / sd).astype(np.float32, copy=False) for x in subj]

def normalize_subject_list(subject_list):
    subj = [coerce_TxV(x) for x in subject_list]
    if HT_NORMALIZE_AVAILABLE:
        try:
            return ht.normalize(subj, normalize="across")
        except Exception:
            pass
    return normalize_across_fallback(subj)

def load_condition_data():
    pieman_data = loadmat(PIEMAN_MAT_PATH)

    data = []
    conds = []

    for c in CONDITIONS:
        next_data = list(
            map(
                lambda i: pieman_data[c][:, i][0],
                np.arange(pieman_data[c].shape[1])
            )
        )
        data.extend(next_data)
        conds.extend([c] * len(next_data))

    conds_array = np.asarray(conds)

    condition_data = {}

    for c in CONDITIONS:
        cur = [data[i] for i in np.where(conds_array == c)[0]]

        if c == "rest" and TRIM_REST_BY > 0:
            cur = array_cutter(cur, TRIM_REST_BY)

        condition_data[c] = normalize_subject_list(cur)

    return condition_data

condition_data = load_condition_data()

for c in CONDITIONS:
    print(
        c,
        "N =", len(condition_data[c]),
        "shape =", np.asarray(condition_data[c][0]).shape
    )

T = np.asarray(condition_data[CONDITIONS[0]][0]).shape[0]
V = np.asarray(condition_data[CONDITIONS[0]][0]).shape[1]

print("T =", T, "| V =", V)
print("Analytical chance for exact timepoint identification =", 1.0 / T)


## PCA orientations

We match the two MS-AA orientations by changing which domain supplies PCA's observations/features.

### Temporal PCA analogue
Input matrices remain \(T \times V\). PCA therefore learns shared **spatial components** over the \(V\) nodes, with subject-specific temporal scores.

This is the direct PCA analogue of **temporal AA**.

### Spatial PCA analogue
Input matrices are transposed to \(V \times T\). PCA therefore learns shared **temporal components** over the \(T\) timepoints, with node-specific scores.

This is the direct PCA analogue of **spatial AA**.

The PCA basis is fit jointly across all subjects and conditions, matching the shared across-condition MS-AA basis.


In [ ]:
# ============================================================
# FIT SHARED PCA BASES
# ============================================================

def orient_for_pca(Xtv, analysis_type):
    Xtv = coerce_TxV(Xtv)

    if analysis_type == "temporal":
        # observations=timepoints, features=nodes
        return Xtv

    if analysis_type == "spatial":
        # observations=nodes, features=timepoints
        return Xtv.T

    raise ValueError("analysis_type must be spatial or temporal")

def pooled_oriented_data(condition_data, analysis_type):
    mats = []

    for condition in CONDITIONS:
        for X in condition_data[condition]:
            mats.append(orient_for_pca(X, analysis_type))

    return np.vstack(mats)

def max_valid_components(analysis_type):
    pooled = pooled_oriented_data(condition_data, analysis_type)
    return int(min(pooled.shape[0], pooled.shape[1]))

pca_models = {}

for analysis_type in ANALYSIS_TYPES:
    pooled = pooled_oriented_data(condition_data, analysis_type)

    requested_max = max(K_VALUES_BY_ANALYSIS[analysis_type])
    valid_max = min(requested_max, min(pooled.shape))

    print("\n", analysis_type.upper())
    print("Pooled PCA matrix:", pooled.shape)
    print("Requested max K:", requested_max)
    print("Valid max K:", valid_max)

    model = PCA(
        n_components=valid_max,
        svd_solver=PCA_SOLVER,
        whiten=WHITEN,
        random_state=RNG_SEED,
    )

    model.fit(pooled)

    pca_models[analysis_type] = model

    print(
        "Cumulative explained variance at max K:",
        model.explained_variance_ratio_.sum()
    )


In [ ]:
# ============================================================
# PCA RECONSTRUCTION
# ============================================================

def reconstruct_pca_subject(
    Xtv,
    analysis_type,
    pca_model,
    K,
    component_indices=None,
):
    """
    Reconstruct a single subject in the original T x V space.

    K controls the fitted representational dimensionality available.
    component_indices optionally selects a subset among the first K components.
    """
    X_oriented = orient_for_pca(Xtv, analysis_type)

    scores_all = pca_model.transform(X_oriented)

    K = int(K)
    if K > scores_all.shape[1]:
        raise ValueError(f"K={K} exceeds fitted PCA dimensionality {scores_all.shape[1]}")

    if component_indices is None:
        idx = np.arange(K)
    else:
        idx = np.asarray(component_indices, dtype=int)
        idx = idx[idx < K]

    # sklearn PCA inverse with only selected coefficients retained
    scores_use = np.zeros_like(scores_all)
    scores_use[:, idx] = scores_all[:, idx]

    Xhat_oriented = scores_use @ pca_model.components_ + pca_model.mean_

    if analysis_type == "temporal":
        Xhat = Xhat_oriented
    else:
        Xhat = Xhat_oriented.T

    # Match notebook 002 reconstruction normalization
    Xhat = (
        Xhat - Xhat.mean(axis=0, keepdims=True)
    ) / (
        Xhat.std(axis=0, keepdims=True) + 1e-8
    )

    return Xhat

def build_pca_recon_stack(
    subjects,
    analysis_type,
    pca_model,
    K,
    component_indices=None,
):
    return np.stack([
        reconstruct_pca_subject(
            X,
            analysis_type,
            pca_model,
            K,
            component_indices=component_indices,
        )
        for X in subjects
    ], axis=0)


## Use the identical cross-subject timepoint-matching decoder from notebook 002

In [ ]:
# ============================================================
# TIMEPOINT IDENTIFICATION
# copied from notebook 002
# ============================================================

def decoder(corrs):
    out = pd.DataFrame({
        "rank": [0.0],
        "accuracy": [0.0],
        "error": [0.0],
    })

    T_local = corrs.shape[0]

    for t in range(T_local):
        decoded_ind = int(np.argmax(corrs[t, :]))

        out.loc[0, "error"] += np.abs(decoded_ind - t) / T_local
        out.loc[0, "accuracy"] += (decoded_ind == t)
        out.loc[0, "rank"] += np.mean(
            (corrs[t, :] <= corrs[t, t]).astype(int)
        )

    out["error"] /= T_local
    out["accuracy"] /= T_local
    out["rank"] /= T_local

    return out

def get_xval_assignments(ndata, nfolds, rng=None):
    rng = np.random.default_rng() if rng is None else rng

    group_assignments = np.zeros(ndata, dtype=int)
    groupsize = int(np.ceil(ndata / nfolds))

    for i in range(1, nfolds):
        inds = np.arange(
            i * groupsize,
            min((i + 1) * groupsize, ndata)
        )
        group_assignments[inds] = i

    rng.shuffle(group_assignments)

    return group_assignments

def run_timepoint_decoding(
    recon_stack,
    nfolds=NFOLDS_DECODE,
    nreps=NREPS_DECODE,
    seed=RNG_SEED,
):
    N, T_local, V_local = recon_stack.shape

    rng = np.random.default_rng(seed)

    rows = []

    for rep in range(nreps):
        fold_ids = get_xval_assignments(
            N,
            nfolds,
            rng=rng,
        )

        for fold in range(nfolds):
            in_mask = fold_ids == fold
            out_mask = ~in_mask

            in_mean = recon_stack[in_mask].mean(axis=0)
            out_mean = recon_stack[out_mask].mean(axis=0)

            corrs = 1.0 - cdist(
                in_mean,
                out_mean,
                metric="correlation",
            )

            corrs = np.nan_to_num(
                corrs,
                nan=0.0,
                posinf=0.0,
                neginf=0.0,
            )

            res = decoder(corrs)
            res["rep"] = rep
            res["fold"] = fold

            rows.append(res)

    return pd.concat(rows, ignore_index=True)

def summarize_decoding_runs(dec_df):
    return {
        "mean": float(dec_df["accuracy"].mean()),
        "err": float(dec_df["accuracy"].std() / np.sqrt(max(len(dec_df), 1))),
        "std": float(dec_df["accuracy"].std()),
        "count": int(len(dec_df)),
        "rank_mean": float(dec_df["rank"].mean()),
        "error_mean": float(dec_df["error"].mean()),
    }


## Raw HTFA benchmark

In [ ]:
# ============================================================
# RAW 700-NODE HTFA TIMEPOINT IDENTIFICATION
# ============================================================

raw_rows = []

if RUN_RAW_HTFA_BASELINE:
    for condition in CONDITIONS:
        subjects = condition_data[condition]

        raw_stack = np.stack([
            (
                X - X.mean(axis=0, keepdims=True)
            ) / (
                X.std(axis=0, keepdims=True) + 1e-8
            )
            for X in subjects
        ], axis=0)

        dec_df = run_timepoint_decoding(raw_stack)
        summary = summarize_decoding_runs(dec_df)

        raw_rows.append({
            "method": "raw_htfa",
            "analysis_type": "raw",
            "condition": condition,
            "K": V,
            "component_ratio": 1.0,
            **summary,
        })

raw_summary_df = pd.DataFrame(raw_rows)

display(raw_summary_df)

raw_summary_df.to_csv(
    OUTPUT_DIR / "raw_htfa_summary.csv",
    index=False,
)


## Full PCA reconstruction across \(K\)

In [ ]:
# ============================================================
# FULL PCA K CURVES
# ============================================================

full_rows = []

for analysis_type in ANALYSIS_TYPES:
    model = pca_models[analysis_type]

    valid_Ks = [
        K for K in K_VALUES_BY_ANALYSIS[analysis_type]
        if K <= model.n_components_
    ]

    for K in valid_Ks:
        print("\n", analysis_type, "K=", K)

        for condition in CONDITIONS:
            subjects = condition_data[condition]

            recon_stack = build_pca_recon_stack(
                subjects,
                analysis_type,
                model,
                K,
                component_indices=None,
            )

            dec_df = run_timepoint_decoding(recon_stack)
            summary = summarize_decoding_runs(dec_df)

            D = V if analysis_type == "spatial" else T

            full_rows.append({
                "method": "PCA",
                "analysis_type": analysis_type,
                "fit_scope": FIT_SCOPE,
                "condition": condition,
                "K": int(K),
                "component_ratio": float(K / D),
                "variance_explained": float(
                    model.explained_variance_ratio_[:K].sum()
                ),
                **summary,
            })

pca_full_df = pd.DataFrame(full_rows)

pca_full_df.to_csv(
    OUTPUT_DIR / "pca_full_summary.csv",
    index=False,
)

display(pca_full_df.head())


## Sparse PCA using the natural variance ordering

Unlike AA, PCA already supplies a canonical ordering: PC1, PC2, ... are ranked by explained variance.

This is useful for the reviewer response because it gives us a **selection-independent sparse PCA baseline**. It asks:

> How much cross-subject temporal information is retained by the first \(m\) variance-maximizing PCA components?

For PCA, this sparse reconstruction is largely independent of nominal \(K\) once \(K \ge m\), because PCA bases are nested. That distinction is itself worth noting when contrasting PCA with AA: AA archetypes can reorganize as \(K\) changes, whereas PCA's leading directions are nested.


In [ ]:
# ============================================================
# VARIANCE-RANKED PCA TOP-M
# ============================================================

variance_topm_rows = []

if RUN_VARIANCE_RANKED_TOPM:

    for analysis_type in ANALYSIS_TYPES:
        model = pca_models[analysis_type]

        valid_Ks = [
            K for K in K_VALUES_BY_ANALYSIS[analysis_type]
            if K <= model.n_components_
        ]

        for K in valid_Ks:

            for top_m in TOP_M_VALUES:
                if top_m > K:
                    continue

                selected = list(range(top_m))

                print(
                    analysis_type,
                    "K=", K,
                    "top_m=", top_m
                )

                for condition in CONDITIONS:
                    subjects = condition_data[condition]

                    recon_stack = build_pca_recon_stack(
                        subjects,
                        analysis_type,
                        model,
                        K,
                        component_indices=selected,
                    )

                    dec_df = run_timepoint_decoding(recon_stack)
                    summary = summarize_decoding_runs(dec_df)

                    D = V if analysis_type == "spatial" else T

                    variance_topm_rows.append({
                        "method": "PCA",
                        "ranking": "explained_variance",
                        "analysis_type": analysis_type,
                        "fit_scope": FIT_SCOPE,
                        "condition": condition,
                        "K": int(K),
                        "component_ratio": float(K / D),
                        "top_m": int(top_m),
                        "selected_components": str(selected),
                        "variance_explained_selected": float(
                            model.explained_variance_ratio_[:top_m].sum()
                        ),
                        **summary,
                    })

pca_variance_topm_df = pd.DataFrame(variance_topm_rows)

pca_variance_topm_df.to_csv(
    OUTPUT_DIR / "pca_variance_ranked_topm_summary.csv",
    index=False,
)

display(pca_variance_topm_df.head())


## Optional: decoding-ranked PCA components

This section is **off by default**.

It mirrors the current AA top-\(m\) logic by decoding individual PCA components, ranking them by the same timepoint-identification statistic, and reconstructing using the highest-ranked components.

However, the reviewers correctly noted that ranking and evaluating on the same data can inflate sparse performance. Therefore this should be treated as a diagnostic comparison only unless both AA and PCA are subsequently changed to use held-out component selection.


In [ ]:
# ============================================================
# OPTIONAL DECODING-RANKED PCA
# ============================================================

pca_per_component_df = pd.DataFrame()
pca_decoding_topm_df = pd.DataFrame()

if RUN_DECODING_RANKED_PCA:

    per_rows = []
    top_rows = []

    for analysis_type in ANALYSIS_TYPES:
        model = pca_models[analysis_type]

        valid_Ks = [
            K for K in K_VALUES_BY_ANALYSIS[analysis_type]
            if K <= model.n_components_
        ]

        for K in valid_Ks:

            # Individual component scores are condition-specific
            condition_rankings = {}

            for condition in CONDITIONS:
                subjects = condition_data[condition]

                this_scores = []

                for component in range(K):
                    recon_stack = build_pca_recon_stack(
                        subjects,
                        analysis_type,
                        model,
                        K,
                        component_indices=[component],
                    )

                    dec_df = run_timepoint_decoding(recon_stack)
                    summary = summarize_decoding_runs(dec_df)

                    per_rows.append({
                        "method": "PCA",
                        "ranking": "individual_timepoint_identification",
                        "analysis_type": analysis_type,
                        "condition": condition,
                        "K": int(K),
                        "component": int(component),
                        **summary,
                    })

                    this_scores.append(
                        (component, summary["mean"])
                    )

                condition_rankings[condition] = [
                    c for c, score in sorted(
                        this_scores,
                        key=lambda x: x[1],
                        reverse=True,
                    )
                ]

            for condition in CONDITIONS:
                subjects = condition_data[condition]
                ranking = condition_rankings[condition]

                for top_m in TOP_M_VALUES:
                    if top_m > K:
                        continue

                    selected = ranking[:top_m]

                    recon_stack = build_pca_recon_stack(
                        subjects,
                        analysis_type,
                        model,
                        K,
                        component_indices=selected,
                    )

                    dec_df = run_timepoint_decoding(recon_stack)
                    summary = summarize_decoding_runs(dec_df)

                    D = V if analysis_type == "spatial" else T

                    top_rows.append({
                        "method": "PCA",
                        "ranking": "individual_timepoint_identification",
                        "analysis_type": analysis_type,
                        "condition": condition,
                        "K": int(K),
                        "component_ratio": float(K / D),
                        "top_m": int(top_m),
                        "selected_components": str(selected),
                        **summary,
                    })

    pca_per_component_df = pd.DataFrame(per_rows)
    pca_decoding_topm_df = pd.DataFrame(top_rows)

    pca_per_component_df.to_csv(
        OUTPUT_DIR / "pca_per_component_decoding.csv",
        index=False,
    )

    pca_decoding_topm_df.to_csv(
        OUTPUT_DIR / "pca_decoding_ranked_topm_summary.csv",
        index=False,
    )

display(pca_per_component_df.head() if len(pca_per_component_df) else pd.DataFrame())
display(pca_decoding_topm_df.head() if len(pca_decoding_topm_df) else pd.DataFrame())


## Load the existing AA results for direct matched plots

In [ ]:
# ============================================================
# LOAD AA RESULTS
# ============================================================

aa_full_parts = []
aa_topm_parts = []

for analysis_type in ANALYSIS_TYPES:

    d = Path(
        AA_OUTPUT_TEMPLATE.format(
            analysis_type=analysis_type
        )
    )

    full_path = d / "full_summary.csv"
    topm_path = d / "topm_summary.csv"

    if full_path.exists():
        x = pd.read_csv(full_path)

        if "analysis_type" not in x.columns:
            x["analysis_type"] = analysis_type

        D = V if analysis_type == "spatial" else T

        x["component_ratio"] = x["K"] / D
        x["method"] = "MS-AA"

        aa_full_parts.append(x)

        print("Loaded:", full_path)

    else:
        print("Missing:", full_path)

    if topm_path.exists():
        x = pd.read_csv(topm_path)

        if "analysis_type" not in x.columns:
            x["analysis_type"] = analysis_type

        D = V if analysis_type == "spatial" else T

        x["component_ratio"] = x["K"] / D
        x["method"] = "MS-AA"

        aa_topm_parts.append(x)

        print("Loaded:", topm_path)

    else:
        print("Missing:", topm_path)

aa_full_df = (
    pd.concat(aa_full_parts, ignore_index=True)
    if aa_full_parts else pd.DataFrame()
)

aa_topm_df = (
    pd.concat(aa_topm_parts, ignore_index=True)
    if aa_topm_parts else pd.DataFrame()
)

print("AA full rows:", len(aa_full_df))
print("AA top-m rows:", len(aa_topm_df))


## Figure 1: Full reconstruction — PCA vs MS-AA vs raw HTFA

In [ ]:
# ============================================================
# FULL-RECONSTRUCTION COMPARISON
# ============================================================

COND_COLORS = {
    "intact": "purple",
    "word": "green",
    "rest": "black",
}

def plot_full_method_comparison(analysis_type):

    fig, ax = plt.subplots(figsize=(8.2, 5.3))

    # PCA
    for condition in CONDITIONS:
        sub = pca_full_df[
            (pca_full_df["analysis_type"] == analysis_type) &
            (pca_full_df["condition"] == condition)
        ].sort_values("component_ratio")

        if len(sub):
            ax.plot(
                sub["component_ratio"],
                sub["mean"],
                linestyle="--",
                linewidth=2,
                color=COND_COLORS[condition],
                label=f"PCA {condition}",
            )

    # AA
    if len(aa_full_df):
        for condition in CONDITIONS:
            sub = aa_full_df[
                (aa_full_df["analysis_type"].astype(str) == analysis_type) &
                (aa_full_df["condition"].astype(str) == condition)
            ].sort_values("component_ratio")

            if len(sub):
                ax.plot(
                    sub["component_ratio"],
                    sub["mean"],
                    linestyle="-",
                    linewidth=2,
                    color=COND_COLORS[condition],
                    label=f"MS-AA {condition}",
                )

    # Raw HTFA reference
    if len(raw_summary_df):
        for condition in CONDITIONS:
            raw = raw_summary_df[
                raw_summary_df["condition"] == condition
            ]
            if len(raw):
                ax.axhline(
                    raw.iloc[0]["mean"],
                    color=COND_COLORS[condition],
                    linestyle=":",
                    linewidth=1.2,
                    alpha=0.7,
                )

    ax.axhline(
        1.0 / T,
        color="gray",
        linestyle=":",
        linewidth=1.5,
        label="chance",
    )

    ax.set_xlabel("Normalized component ratio")
    ax.set_ylabel("Cross-subject timepoint-identification accuracy")
    ax.set_title(
        f"{analysis_type.capitalize()} orientation: "
        "MS-AA vs PCA"
    )

    ax.legend(
        frameon=False,
        fontsize=8,
        ncol=2,
    )

    plt.tight_layout()
    save_current_fig(
        f"full_MSAA_vs_PCA_{analysis_type}"
    )
    plt.show()
    plt.close()

for analysis_type in ANALYSIS_TYPES:
    plot_full_method_comparison(analysis_type)


## Figure 2: Sparse performance relative to each method's full reconstruction

This may be the most reviewer-useful comparison.

For each \(K,m\), calculate:

\[
\mathrm{recovery\ fraction}
=
\frac{\mathrm{top\text{-}m\ accuracy}-\mathrm{chance}}
{\mathrm{full\ reconstruction\ accuracy}-\mathrm{chance}}.
\]

A value near 1 means the sparse representation recovers nearly all above-chance information available to the full representation.

This lets us ask whether AA provides especially efficient sparse representations without requiring AA to beat PCA in absolute accuracy.


In [ ]:
# ============================================================
# RECOVERY FRACTION
# ============================================================

CHANCE = 1.0 / T

def add_recovery_fraction(topm_df, full_df):
    out = []

    for _, r in topm_df.iterrows():

        match = full_df[
            (full_df["analysis_type"].astype(str) == str(r["analysis_type"])) &
            (full_df["condition"].astype(str) == str(r["condition"])) &
            (full_df["K"].astype(int) == int(r["K"]))
        ]

        if len(match) == 0:
            continue

        full_acc = float(match.iloc[0]["mean"])
        sparse_acc = float(r["mean"])

        denom = full_acc - CHANCE

        recovery = (
            (sparse_acc - CHANCE) / denom
            if denom > 1e-12
            else np.nan
        )

        rr = r.to_dict()
        rr["full_mean"] = full_acc
        rr["chance"] = CHANCE
        rr["recovery_fraction"] = recovery

        out.append(rr)

    return pd.DataFrame(out)

pca_recovery_df = add_recovery_fraction(
    pca_variance_topm_df,
    pca_full_df,
)

if len(aa_topm_df) and len(aa_full_df):
    aa_recovery_df = add_recovery_fraction(
        aa_topm_df,
        aa_full_df,
    )
else:
    aa_recovery_df = pd.DataFrame()

display(pca_recovery_df.head())
display(aa_recovery_df.head() if len(aa_recovery_df) else pd.DataFrame())


In [ ]:
# ============================================================
# PLOT RECOVERY FRACTION FOR SELECTED TOP-M
# ============================================================

def plot_recovery_fraction(
    analysis_type,
    condition="intact",
    top_m=5,
):

    fig, ax = plt.subplots(figsize=(7.4, 4.8))

    pca = pca_recovery_df[
        (pca_recovery_df["analysis_type"] == analysis_type) &
        (pca_recovery_df["condition"] == condition) &
        (pca_recovery_df["top_m"] == top_m)
    ].sort_values("component_ratio")

    if len(pca):
        ax.plot(
            pca["component_ratio"],
            pca["recovery_fraction"],
            linestyle="--",
            marker="s",
            linewidth=2,
            label="PCA: variance-ranked",
        )

    if len(aa_recovery_df):
        aa = aa_recovery_df[
            (aa_recovery_df["analysis_type"].astype(str) == analysis_type) &
            (aa_recovery_df["condition"].astype(str) == condition) &
            (aa_recovery_df["top_m"].astype(int) == top_m)
        ].sort_values("component_ratio")

        if len(aa):
            ax.plot(
                aa["component_ratio"],
                aa["recovery_fraction"],
                linestyle="-",
                marker="o",
                linewidth=2,
                label="MS-AA: decoding-ranked",
            )

    ax.axhline(
        1.0,
        color="gray",
        linestyle=":",
        linewidth=1,
    )

    ax.set_xlabel("Normalized component ratio")
    ax.set_ylabel("Fraction of above-chance full-model performance recovered")
    ax.set_title(
        f"{analysis_type.capitalize()} | {condition} | top-{top_m}"
    )

    ax.legend(frameon=False)

    plt.tight_layout()

    save_current_fig(
        f"recovery_fraction_{analysis_type}_{condition}_top{top_m}"
    )

    plt.show()
    plt.close()

for analysis_type in ANALYSIS_TYPES:
    for condition in ["intact", "word"]:
        for top_m in [3, 5, 10]:
            plot_recovery_fraction(
                analysis_type,
                condition=condition,
                top_m=top_m,
            )


## Figure 3: Absolute sparse performance

This is deliberately separate from the recovery-fraction plot. It prevents a representation with poor full-model performance from looking artificially good simply because its sparse reconstruction recovers a large fraction of a low ceiling.


In [ ]:
# ============================================================
# ABSOLUTE TOP-M PERFORMANCE
# ============================================================

def plot_absolute_sparse_comparison(
    analysis_type,
    condition="intact",
    top_m=5,
):

    fig, ax = plt.subplots(figsize=(7.4, 4.8))

    pca = pca_variance_topm_df[
        (pca_variance_topm_df["analysis_type"] == analysis_type) &
        (pca_variance_topm_df["condition"] == condition) &
        (pca_variance_topm_df["top_m"] == top_m)
    ].sort_values("component_ratio")

    if len(pca):
        ax.errorbar(
            pca["component_ratio"],
            pca["mean"],
            yerr=pca["err"],
            linestyle="--",
            marker="s",
            capsize=3,
            linewidth=2,
            label="PCA: variance-ranked",
        )

    if len(aa_topm_df):
        aa = aa_topm_df[
            (aa_topm_df["analysis_type"].astype(str) == analysis_type) &
            (aa_topm_df["condition"].astype(str) == condition) &
            (aa_topm_df["top_m"].astype(int) == top_m)
        ].sort_values("component_ratio")

        if len(aa):
            ax.errorbar(
                aa["component_ratio"],
                aa["mean"],
                yerr=aa["err"] if "err" in aa.columns else None,
                linestyle="-",
                marker="o",
                capsize=3,
                linewidth=2,
                label="MS-AA: decoding-ranked",
            )

    ax.axhline(
        CHANCE,
        color="gray",
        linestyle=":",
        linewidth=1.5,
        label="chance",
    )

    ax.set_xlabel("Normalized component ratio")
    ax.set_ylabel("Cross-subject timepoint-identification accuracy")
    ax.set_title(
        f"{analysis_type.capitalize()} | {condition} | top-{top_m}"
    )

    ax.legend(frameon=False)

    plt.tight_layout()

    save_current_fig(
        f"absolute_sparse_MSAA_vs_PCA_{analysis_type}_{condition}_top{top_m}"
    )

    plt.show()
    plt.close()

for analysis_type in ANALYSIS_TYPES:
    for condition in ["intact", "word"]:
        for top_m in [3, 5, 10]:
            plot_absolute_sparse_comparison(
                analysis_type,
                condition=condition,
                top_m=top_m,
            )


## Compact summary table for the response letter

In [ ]:
# ============================================================
# PAPER / RESPONSE SUMMARY TABLE
# ============================================================

summary_rows = []

for analysis_type in ANALYSIS_TYPES:
    for condition in ["intact", "word"]:

        # Best PCA full
        p = pca_full_df[
            (pca_full_df["analysis_type"] == analysis_type) &
            (pca_full_df["condition"] == condition)
        ]

        if len(p):
            r = p.loc[p["mean"].idxmax()]
            summary_rows.append({
                "method": "PCA",
                "analysis_type": analysis_type,
                "condition": condition,
                "summary_type": "best_full",
                "K": int(r["K"]),
                "top_m": np.nan,
                "accuracy": r["mean"],
                "component_ratio": r["component_ratio"],
            })

        # Best AA full
        if len(aa_full_df):
            a = aa_full_df[
                (aa_full_df["analysis_type"].astype(str) == analysis_type) &
                (aa_full_df["condition"].astype(str) == condition)
            ]

            if len(a):
                r = a.loc[a["mean"].idxmax()]
                summary_rows.append({
                    "method": "MS-AA",
                    "analysis_type": analysis_type,
                    "condition": condition,
                    "summary_type": "best_full",
                    "K": int(r["K"]),
                    "top_m": np.nan,
                    "accuracy": r["mean"],
                    "component_ratio": r["component_ratio"],
                })

        # Best sparse PCA among top 3/5/10
        ps = pca_variance_topm_df[
            (pca_variance_topm_df["analysis_type"] == analysis_type) &
            (pca_variance_topm_df["condition"] == condition) &
            (pca_variance_topm_df["top_m"].isin([3, 5, 10]))
        ]

        if len(ps):
            r = ps.loc[ps["mean"].idxmax()]
            summary_rows.append({
                "method": "PCA",
                "analysis_type": analysis_type,
                "condition": condition,
                "summary_type": "best_sparse_3_5_10",
                "K": int(r["K"]),
                "top_m": int(r["top_m"]),
                "accuracy": r["mean"],
                "component_ratio": r["component_ratio"],
            })

        if len(aa_topm_df):
            aa = aa_topm_df[
                (aa_topm_df["analysis_type"].astype(str) == analysis_type) &
                (aa_topm_df["condition"].astype(str) == condition) &
                (aa_topm_df["top_m"].astype(int).isin([3, 5, 10]))
            ]

            if len(aa):
                r = aa.loc[aa["mean"].idxmax()]
                summary_rows.append({
                    "method": "MS-AA",
                    "analysis_type": analysis_type,
                    "condition": condition,
                    "summary_type": "best_sparse_3_5_10",
                    "K": int(r["K"]),
                    "top_m": int(r["top_m"]),
                    "accuracy": r["mean"],
                    "component_ratio": r["component_ratio"],
                })

paper_summary_df = pd.DataFrame(summary_rows)

paper_summary_df.to_csv(
    OUTPUT_DIR / "pca_msaa_response_summary.csv",
    index=False,
)

display(paper_summary_df)


# How I would interpret this benchmark

The PCA comparison should **not** be framed as requiring AA to win on raw timepoint-identification accuracy.

Three outcomes are scientifically useful:

### 1. PCA and AA have similar full-model performance
This would show that both preserve the same broad stimulus-locked structure. The AA contribution then lies in the geometry and interpretability of the sparse representation.

### 2. PCA performs better at full reconstruction, but AA achieves strong sparse recovery
This is also a useful result. PCA is explicitly optimized for variance preservation, whereas AA is optimized to characterize extreme data-supported configurations. Comparable information preservation from a small number of interpretable archetypes would support the AA motivation.

### 3. AA outperforms PCA in sparse recovery
This would provide stronger evidence that condition-relevant information is particularly concentrated in archetypal representations.

The most important comparison for the manuscript is therefore likely to be **absolute sparse performance together with fraction of full-model performance recovered**, rather than simply asking which method has the single highest decoding value.

## Important limitation

The current MS-AA top-\(m\) curves use decoding-based archetype selection on the same data used for evaluation. Reviewer 1 correctly identified this as a possible source of selection bias.

Therefore:

- use the **variance-ranked PCA result** as the clean initial PCA baseline;
- do not overinterpret a direct variance-ranked-PCA vs decoding-ranked-AA sparse comparison;
- after implementing held-out AA ranking, add a corresponding held-out PCA ranking analysis if needed.

That revised held-out analysis would be the strongest final comparison.
